<a href="https://colab.research.google.com/github/mohammadanwarx/Machine_Learning_Land-Cover_Classification_in_Sudan/blob/main/Supervised%20Land%20Cover%20Classification%20of%20Sudan%20Using%20Google%20Earth%20Engine%20Python%20API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

  # <center>  Big Geodata Processing: Machine Learning Assignment </center>


<h4>----------------------------------------------------- </h4>

**Name:** (MO) Mohamedelmustafa Anwarelsadat Gasmalla  
**Student number:** s3708748
<h4>----------------------------------------------------- </h4>



## connect to GEE APIs

In [26]:
import ee

try:
  ee.Authenticate()
  ee.Initialize(project='ee-mohammadanwarx9922')
  print('Earth Engine authenticated and initialized successfully!')
except ee.EEException as e:
  print(f'Earth Engine initialization failed: {e}')
except Exception as e:
  print(f'An unexpected error occurred: {e}')

Earth Engine authenticated and initialized successfully!


In [24]:
Sudan = [(21.93681, 8.61972971293, 38.4100899595, 22.0)] # study area

In [37]:
# Define the Area of Interest (AOI) as an Earth Engine geometry.
# The AOI is given as (min_lon, min_lat, max_lon, max_lat).
min_lon, min_lat, max_lon, max_lat = Sudan[0]
geometry = ee.Geometry.Rectangle([min_lon, min_lat, max_lon, max_lat])

# Filter the Sentinel-2 image collection.
# We are looking for 'COPERNICUS/S2_SR_HARMONIZED' which is the Surface Reflectance product.
collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')\
    .filterDate('2025-05-01', '2025-09-30')\
    .filterBounds(geometry)\
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))

# You might want to add a cloud cover filter, for example:
# .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))

# Print the number of images in the collection to verify.
print(f'Number of images in the collection: {collection.size().getInfo()}')

# Get a representative image from the collection (e.g., the first one)
# and print its ID to confirm.
first_image = collection.first()
if first_image:
  print(f'ID of the first image: {first_image.id().getInfo()}')
else:
  print('No images found in the collection for the specified criteria.')

Number of images in the collection: 10802
ID of the first image: 20250501T080609_20250501T081819_T35PRK


In [38]:
# Calculate the median composite of the image collection.
median_image = collection.median()

# Print the median image to inspect its properties.
print('Median image:', median_image.getInfo())

Median image: {'type': 'Image', 'bands': [{'id': 'B1', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B2', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B3', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B4', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B5', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B6', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B7', 'data_type': {'type':

In [39]:
import folium

# Define visualization parameters for True Color Composite (B4, B3, B2)
true_color_vis = {
    'min': 0,
    'max': 3000,
    'bands': ['B4', 'B3', 'B2'],
}

# Define visualization parameters for False Color Composite (B8, B4, B3) - NIR, Red, Green
false_color_vis = {
    'min': 0,
    'max': 3000,
    'bands': ['B8', 'B4', 'B3'], # NIR, Red, Green. Vegetation appears red.
}

# Calculate the centroid of the geometry to center the map
centroid = geometry.centroid().coordinates().getInfo()

# Create a Folium map object.
my_map = folium.Map(location=[centroid[1], centroid[0]], zoom_start=6)

# Get a Google Earth Engine tile URL for the True Color image
true_color_map_id = median_image.getMapId(true_color_vis)
true_color_tile_url = true_color_map_id['tile_fetcher'].url_format

# Add the GEE True Color layer to the Folium map.
folium.TileLayer(
    tiles=true_color_tile_url,
    attr='Google Earth Engine',
    name='True Color Composite (Median)',
    overlay=True,
    control=True
).add_to(my_map)

# Get a Google Earth Engine tile URL for the False Color image
false_color_map_id = median_image.getMapId(false_color_vis)
false_color_tile_url = false_color_map_id['tile_fetcher'].url_format

# Add the GEE False Color layer to the Folium map.
folium.TileLayer(
    tiles=false_color_tile_url,
    attr='Google Earth Engine',
    name='False Color Composite (Median)',
    overlay=True,
    control=True
).add_to(my_map)

# Add a layer control panel to the map.
my_map.add_child(folium.LayerControl())

# Display the map.
display(my_map)

## Define Training Samples


